# 未来創生センターの論文の指標を CSV に書き出す（2026-10-07）

今井先生の依頼（2026-10-07）:「SciSciNet から抽出した新規性指標など、前回の報告資料でまとめたデータのうち、未来創生センターに関するものだけを CSV に」。

- 前回の報告資料: `docs/slides/toyota_novelty_progress.pptx`（2026-09-28）
- 数字の出どころは資料と同じ表（`docs/slides/make_numbers_progress_2026-09.py` が読む表）。新しい計算はしない
- 1行 = 公式サイトの掲載1件（154行。報告資料4ページの「サイト掲載 154」と同じ数え方）
- 最後のセルで、書き出した CSV から報告資料4ページ・8ページの数字を計算し直して照合する

出力:
- `docs/exports/frontier_research_center_metrics.csv` … データ本体
- `docs/exports/frontier_research_center_columns.csv` … 列の説明

どちらも Excel で文字化けしないよう、UTF-8（BOM 付き）で書き出す。

注: リポジトリ内のデータでは組織名を「トヨタ自動車未来創成センター」と書いているが、正式名称は「未来創生センター」。

In [1]:
from pathlib import Path

import pandas as pd

# リポジトリの最上位（AGENTS.md がある場所）を探す
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
DD = ROOT / "data" / "derived"
OUT_DIR = ROOT / "docs" / "exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ORG = "トヨタ自動車未来創成センター"  # データ内の表記

links = pd.read_csv(DD / "openalex_linking" / "openalex_links.csv")    # サイト掲載1件1行（919行）
rowmap = pd.read_csv(DD / "papers" / "rowid_map.csv")                  # 掲載 → 論文ID(paper_uid)
papers = pd.read_csv(DD / "papers" / "papers.csv")                     # 論文1件1行（重複掲載をまとめたもの）
ana = pd.read_csv(DD / "analysis" / "paper_analysis.csv")              # SciSciNet の値と説明変数（work 単位）
pct = pd.read_csv(DD / "analysis" / "paper_percentiles.csv")           # 一般論文の中での順位
print(len(links), len(rowmap), len(papers), len(ana), len(pct))

919 919 917 852 671


## 掲載ごとの表を作る

`openalex_links.csv` の行番号（`row_id`）と `rowid_map.csv` で論文ID（`paper_uid`）を付け、その ID で SciSciNet の値・順位をつなぐ。
同じ論文がサイトに2回載っている場合は、2行とも同じ値になる。

In [2]:
s = links[links["source_org"] == ORG].copy()
s = s.merge(rowmap[["row_id", "paper_uid"]], on="row_id", how="left", validate="one_to_one")
s = s.sort_values("row_id").reset_index(drop=True)
assert s["paper_uid"].notna().all()

# 分析の表（ana）は OpenAlex の論文1件につき1行。各論文はどれか1つの組織に割り当てられている
a = ana[ana["paper_uid"].isin(s["paper_uid"])]
assert (a["source_org"] == ORG).all(), "未来創生センターの掲載が別組織に割り当てられている"
s = s.merge(a, on="paper_uid", how="left", suffixes=("", "_ana"))
s = s.merge(pct, on="paper_uid", how="left", suffixes=("", "_pct"))
s = s.merge(papers[["paper_uid", "openalex_doi"]], on="paper_uid", how="left")

# サイト側の OpenAlex ID と分析の表の ID が一致することを確認
both = s["openalex_id"].notna() & s["work_id"].notna()
assert (s.loc[both, "openalex_id"] == s.loc[both, "work_id"]).all()
print(len(s), "行")

154 行


## 備考の列

- サイト内の重複掲載（同じ論文が2回載っている）
- 他の組織のサイトにも載っている論文。資料の組織別の集計では、これらは未来創生センター側に数えている（上の assert で確認済み）

In [3]:
notes = {}
for uid, g in s.groupby("paper_uid"):
    if len(g) > 1:
        nums = "と".join(str(i + 1) for i in g.index)
        for i in g.index:
            notes.setdefault(i, []).append(f"サイト内の重複掲載（掲載順{nums}は同じ論文）")

others = papers[(papers["source_org"] != ORG) & papers["work_id"].isin(s["openalex_id"].dropna())]
for i, wid in s["openalex_id"].items():
    orgs = sorted(set(others.loc[others["work_id"] == wid, "source_org"]))
    if orgs:
        notes.setdefault(i, []).append("、".join(orgs) + "のサイトにも掲載")

s["備考"] = pd.Series({i: "。".join(v) for i, v in notes.items()})
s.loc[s["備考"].notna(), ["title", "備考"]]

,title,備考
27,Wasserstein Non-Negative Matrix Factorization ...,サイト内の重複掲載（掲載順28と39は同じ論文）
29,Subjective mapping of indoor plants based on l...,豊田中央研究所のサイトにも掲載
38,Wasserstein Non-Negative Matrix Factorization ...,サイト内の重複掲載（掲載順28と39は同じ論文）
74,Cooperative Transportation using Multiple Sing...,豊田中央研究所のサイトにも掲載
86,An In Vitro Mixed Infection Model With Commens...,豊田中央研究所のサイトにも掲載
145,Phenotypic diversification by enhanced genome ...,豊田中央研究所のサイトにも掲載


## 書き出す列を作る

- 順位は、同じ分野・同じ年の一般論文の中での位置（0〜100、50が中央）。比べる相手が30件未満のときは空欄
- 非慣習性の順位は、資料と同じく 100 から引いて「高いほど珍しい組み合わせを含む」向きにそろえる
- 順位は2通り載せる。報告資料9ページ（要因の検証）は「分野・年」、8ページ（組織別）は「分野・年・参考文献数・著者数」

In [4]:
def yes_no(x):
    return x.map({True: "はい", False: "いいえ"})


def ari_nashi(x):
    # 値がない（OpenAlex で特定できていない等）ときは空欄のまま
    return x.map(lambda v: "" if pd.isna(v) else ("あり" if bool(v) else "なし"))


matched = s["matched"].astype(bool)
has_ss = s["ss_year"].notna() & (s["year_openalex"] >= 2000)

out = pd.DataFrame({
    "掲載順": range(1, len(s) + 1),
    "タイトル（サイト掲載）": s["title"],
    "著者（サイト掲載）": s["authors_raw"],
    "掲載先（サイト掲載）": s["venue"],
    "出版年（サイト掲載）": s["year"].astype("Int64"),
    "DOI": s["openalex_doi"].fillna(s["doi_norm"]).str.replace("https://doi.org/", "", regex=False),
    "OpenAlex ID": s["openalex_id"].str.replace("https://openalex.org/", "", regex=False),
    "OpenAlexで特定": yes_no(matched),
    "SciSciNetに値あり": yes_no(has_ss),
    "新規性の値あり": yes_no(has_ss & s["ss_novelty"].notna()),
    "備考": s["備考"],
    "出版年（OpenAlex）": s["year_openalex"].astype("Int64"),
    "分野（OpenAlex）": s["primary_field"],
    "定石らしさ（zスコアの中央値）": s["ss_conventionality"],
    "zスコアの下位10パーセンタイル値": s["ss_novelty"],
    "破壊性": s["ss_disruption"],
    "10年被引用": s["ss_c10"],
    "著者数": s["ss_team_size"].astype("Int64"),
    "参考文献数": s["referenced_works_count"].astype("Int64"),
})
for cond, suf in [("分野・年", ""), ("分野・年・参考文献数・著者数", "_ref_team")]:
    out[f"非慣習性の順位（{cond}）"] = (100 - s[f"ss_novelty_pct{suf}"]).round(1)
    out[f"定石らしさの順位（{cond}）"] = s[f"ss_conventionality_pct{suf}"].round(1)
    out[f"破壊性の順位（{cond}）"] = s[f"ss_disruption_pct{suf}"].round(1)
    out[f"10年被引用の順位（{cond}）"] = s[f"ss_c10_pct{suf}"].round(1)
out["分野の偏り"] = s["ref_field_hhi"].round(3)
out["国際共同"] = ari_nashi(s["is_international"])
out["産学連携"] = ari_nashi(s["is_industry_academia_oa"])
out["組織をまたぐ共同"] = ari_nashi(s["is_cross_org"])

# OpenAlex で特定できていない掲載は、分析の表の値を持たない
assert out.loc[~matched, "著者数"].isna().all()
out.to_csv(OUT_DIR / "frontier_research_center_metrics.csv", index=False, encoding="utf-8-sig")
out.shape

(154, 31)

## 列の説明

In [5]:
COLS = [
    ("掲載順", "トヨタ自動車 未来創生センターの論文一覧サイトでの掲載順", "公式サイト（2026-07-27 取得）"),
    ("タイトル（サイト掲載）", "サイトに書かれているタイトル", "公式サイト"),
    ("著者（サイト掲載）", "サイトに書かれている著者", "公式サイト"),
    ("掲載先（サイト掲載）", "サイトに書かれている掲載誌・会議名", "公式サイト"),
    ("出版年（サイト掲載）", "サイトに書かれている出版年", "公式サイト"),
    ("DOI", "論文の DOI（OpenAlex の値。無ければサイトの値）", "OpenAlex・公式サイト"),
    ("OpenAlex ID", "OpenAlex 上の論文の ID。空欄は OpenAlex で特定できなかった論文", "OpenAlex"),
    ("OpenAlexで特定", "タイトル等の照合で OpenAlex 上の論文を特定できたか（報告資料4ページの「OpenAlex で特定」）", "OpenAlex"),
    ("SciSciNetに値あり", "SciSciNet に収録されているか（報告資料4ページの「SciSciNet に値あり」）。SciSciNet は2024年までの論文を収録", "SciSciNet"),
    ("新規性の値あり", "SciSciNet に新規性の値があるか（報告資料4ページの「新規性の値あり」）。参考文献の掲載誌が十分に分からない論文には値がない", "SciSciNet"),
    ("備考", "サイト内の重複掲載、他組織のサイトにも載っている論文。他組織にも載っている論文は、報告資料では未来創生センター側に数えている", "公式サイト"),
    ("出版年（OpenAlex）", "OpenAlex 上の出版年。比較に使う「同じ年」はこの年", "OpenAlex"),
    ("分野（OpenAlex）", "OpenAlex の26分野のうち論文の主分野。比較に使う「同じ分野」はこの分野", "OpenAlex"),
    ("定石らしさ（zスコアの中央値）", "参考文献の掲載誌の組ごとに、一緒に引用される頻度を無作為の場合と比べた zスコアを求め、その中央値をとったもの。高いほど、よくある組が中心", "SciSciNet"),
    ("zスコアの下位10パーセンタイル値", "上の zスコアの下位10パーセンタイル値（非慣習性の元の値）。低いほど珍しい組み合わせを含む", "SciSciNet"),
    ("破壊性", "Wu et al. 2019 の disruption。その論文を引用する後続論文のうち、その論文の参考文献を引用しないものが多いほど高い（−1〜1）。引用されていない論文は0", "SciSciNet"),
    ("10年被引用", "出版後10年間の被引用数。2015年以降の論文は、SciSciNet のデータの時点までの被引用数", "SciSciNet"),
    ("著者数", "論文の著者の人数。0 は SciSciNet に著者の記録がない論文（OpenAlex では著者がいる。2024年の論文に多い）", "SciSciNet"),
    ("参考文献数", "参考文献の本数", "OpenAlex"),
    ("非慣習性の順位（分野・年）", "同じ分野・同じ年の一般論文（無作為に最大3,000件）の中での順位（0〜100、50が中央）。100 から引いて、高いほど珍しい組み合わせを含む向きにしてある。比べる相手が30件未満のときは空欄。報告資料9ページの検証に使った値", "SciSciNet から計算"),
    ("定石らしさの順位（分野・年）", "非慣習性の順位（分野・年）と同じ求め方。高いほど、よくある組が中心", "SciSciNet から計算"),
    ("破壊性の順位（分野・年）", "同じ求め方。高いほど破壊的", "SciSciNet から計算"),
    ("10年被引用の順位（分野・年）", "同じ求め方。高いほど多く引用されている", "SciSciNet から計算"),
    ("非慣習性の順位（分野・年・参考文献数・著者数）", "比べる一般論文を、さらに参考文献数（±35%）と著者数（±1人）が近いものに絞ったときの順位。この4列は報告資料8ページの組織別の表に使った値", "SciSciNet から計算"),
    ("定石らしさの順位（分野・年・参考文献数・著者数）", "同上", "SciSciNet から計算"),
    ("破壊性の順位（分野・年・参考文献数・著者数）", "同上", "SciSciNet から計算"),
    ("10年被引用の順位（分野・年・参考文献数・著者数）", "同上", "SciSciNet から計算"),
    ("分野の偏り", "参考文献の分野ごとの割合の2乗和（1に近いほど1分野に集中）。報告資料9ページで使用", "OpenAlex"),
    ("国際共同", "著者の所属機関の国が2か国以上か。報告資料9ページで使用", "OpenAlex"),
    ("産学連携", "著者の所属機関に教育機関（大学など）と企業の両方を含むか。報告資料9ページで使用", "OpenAlex"),
    ("組織をまたぐ共同", "著者の所属にトヨタ系4組織のうち2組織以上を含むか。報告資料9ページで使用", "OpenAlex 他"),
]
cols = pd.DataFrame(COLS, columns=["列名", "内容", "出典"])
assert list(cols["列名"]) == list(out.columns), set(cols["列名"]) ^ set(out.columns)
cols.to_csv(OUT_DIR / "frontier_research_center_columns.csv", index=False, encoding="utf-8-sig")
len(cols)

31

## 資料との照合

書き出した CSV を読み直して、報告資料4ページ（件数）と8ページ（組織別の順位の中央値）の数字を計算し直す。
8ページの中央値は、同じ論文が2行ある分を1行にしてから計算する（資料は論文単位で集計している）。

In [6]:
c = pd.read_csv(OUT_DIR / "frontier_research_center_metrics.csv", encoding="utf-8-sig")

slide4 = {"サイト掲載": len(c),
          "OpenAlex で特定": int((c["OpenAlexで特定"] == "はい").sum()),
          "SciSciNet に値あり": int((c["SciSciNetに値あり"] == "はい").sum()),
          "新規性の値あり": int((c["新規性の値あり"] == "はい").sum())}
print("報告資料4ページ（154 / 131 / 93 / 69）:", slide4)
assert list(slide4.values()) == [154, 131, 93, 69]

u = c.drop_duplicates("OpenAlex ID")
cond = "分野・年・参考文献数・著者数"
slide8 = {}
for lab in ["非慣習性", "定石らしさ", "破壊性", "10年被引用"]:
    col = u[f"{lab}の順位（{cond}）"]
    slide8[lab] = (round(col.median(), 1), int(col.notna().sum()))
print("報告資料8ページ（59.2 / 48.0 / 52.0 / 48.3）: (中央値, 件数)", slide8)
assert [v[0] for v in slide8.values()] == [59.2, 48.0, 52.0, 48.3]

報告資料4ページ（154 / 131 / 93 / 69）: {'サイト掲載': 154, 'OpenAlex で特定': 131, 'SciSciNet に値あり': 93, '新規性の値あり': 69}
報告資料8ページ（59.2 / 48.0 / 52.0 / 48.3）: (中央値, 件数) {'非慣習性': (np.float64(59.2), 63), '定石らしさ': (np.float64(48.0), 63), '破壊性': (np.float64(52.0), 64), '10年被引用': (np.float64(48.3), 64)}
